# Quixo
---

<img style="float:center" src="../images/quixo.jpg" alt="drawing" width="200"/>

In [2]:
import pickle
from random import randint, random, choice
from game import Game, Move, Player
from tqdm import trange
from typing import Literal
import numpy as np

## Reinforcement Learning Player
---

This class represents a player that uses Reinforcement Learning to make decisions in Quixo.

Attributes:
- ``epochs`` (int): The number of training epochs.
- ``alpha`` (float): The learning rate.
- ``discount_factor`` (float): The discount factor of the Bellman equation.
- ``min_exploration_rate`` (float): The minimum exploration rate during training.
- ``exploration_decay_rate`` (float): The rate at which the exploration rate decays during training.
- ``opponent`` (Player): The opponent player.
- ``states`` (list): A list to store the states visited during a game.
- ``state_value`` (dict): A dictionary to store the value of each state.

Methods:
- ``give_rew(reward)``: Placeholder method for giving reward to the player.
- ``add_state(state)``: Adds to the ``states`` array the state that a player has seen during a game.
- ``reset()``: Reset the ``states`` array to be able to start a new game.
- ``choose_action(game)``: Chooses an action to take based on the current game state that can be random or based on the value of the dictionary. It takes from the dictionary, for each possible move, the value associated with the state of the board with the move performed. The maximum value will be the move to execute. We use the following recursive (bellman equation) formula to compute the state-value table: 
$$
V(S_t) \leftarrow V(S_t) + \alpha * (\gamma * V(S_t +1) - V(S_t))
$$
- ``update_state_value_table(reward)``: Updates the values of the ``states_value`` dictionary based on the states that the player has seen during the game and the reward that they have provided.
- ``game_reward(player)``: Calculates the reward for the player in the current game.
- ``train()``: Trains the player using reinforcement learning.
- ``save_policy(name)``: Saves the state value table to a file.
- ``load_policy(file)``: Loads the state value table from a file.

In [4]:
class RLPlayer(Player):
    def __init__(self, epochs: int,
                 alpha: float,
                 discount_factor: float,
                 min_exploration_rate: float,
                 exploration_decay_rate: float,
                 opponent: 'Player') -> None:
        
        super().__init__()
        self.epochs = epochs
        self.alpha = alpha
        self.discount_factor = discount_factor
        self.exploration_rate = 1
        self.min_exploration_rate = min_exploration_rate
        self.exploration_decay_rate = exploration_decay_rate
        self.opponent = opponent
        self.states=[]
        self.state_value = {}
    
    def give_rew(self, reward):
        pass
    
    def add_state(self, state):
        self.states.append(state)
        
    def reset(self):
        self.states = []
    
    def choose_action(self, game: Game) -> tuple[tuple[int, int], Move]:
        available_moves = game.get_possible_moves(game.get_current_player())
        if random() < self.exploration_rate:  # do exploration
            return choice(available_moves)
        else:  # do exploitation
            value_max = -999
            for move in available_moves:
                tmp = game.get_board()
                game.make_move(move[0], move[1])
                next_status = game.convert_matrix_board_to_tuple(game.get_board())
                game.set_board(tmp)
                value = 0 if self.state_value.get(next_status) is None else self.state_value.get(next_status)
                if value > value_max:
                    value_max = value
                    action = move
        return action
        
    def update_state_value_table(self, reward):
        for st in reversed(self.states):
            if self.state_value.get(st) is None:
                self.state_value[st] = 0
            current_q_value = self.state_value[st]
            reward = current_q_value + self.alpha * (self.discount_factor * reward - current_q_value)
            self.state_value[st] = reward
        
    def game_reward(self, player: 'RLPlayer')-> Literal[-10, 0, 10]:
        if self == player:
            return 10
        else:
            return -10
    
    def train(self) -> None:
        
        all_rewards = []
        # define how many episodes to run
        pbar = trange(self.epochs)
        # define the players
        players = (self, self.opponent)
        
        for epochs in pbar:
            game = Game()
            rewards = 0
            winner = -1
            players = (players[1], players[0])
            player_idx = 1
            
            while winner < 0:
                # change player
                player_idx = (player_idx + 1) % 2
                player = players[player_idx]
                game.switch_player()
                
                ok = False
                if self == player:
                    while not ok:
                        from_pos, slide = self.choose_action(game)
                        ok = game.make_move(from_pos, slide)
                        state_after_move = game.convert_matrix_board_to_tuple(game.get_board())
                        self.add_state(state_after_move)
                        
                else:
                    while not ok:
                        from_pos, slide = player.choose_action(game)
                        ok = game.make_move(from_pos, slide)
                winner = game.check_winner()
            
            # update the exploration rate
            self.exploration_rate = np.clip(
                np.exp(-self.exploration_decay_rate * epochs), self.min_exploration_rate, 1
            )
            
            reward = self.game_reward(player)
            self.update_state_value_table(reward)
            rewards += reward
            self.reset()
            all_rewards.append(rewards)
            pbar.set_description(f'rewards value: {rewards}, current exploration rate: {self.exploration_rate:2f}')
            
        print(f'** Last 1_000 episodes - Mean rewards value: {sum(all_rewards[-1_000:]) / 1_000:.2f} **')
        print(f'** Last rewards value: {all_rewards[-1]:} **')

    def save_policy(self, name):
        fw = open(name, 'wb')
        pickle.dump(self.state_value, fw)
        fw.close()

    def load_policy(self, file):
        fr = open(file, 'rb')
        self.state_value = pickle.load(fr)
        fr.close()

## Random Player Definition

In [7]:
class RandomPlayer(Player):
    def __init__(self) -> None:
        super().__init__()

    def choose_action(self, game: 'Game') -> tuple[tuple[int, int], Move]:
        from_pos = (randint(0, 4), randint(0, 4))
        move = choice([Move.TOP, Move.BOTTOM, Move.LEFT, Move.RIGHT])
        return from_pos, move

    def give_rew(self, reward):
        pass

    def add_state(self, s):
        pass

## Human Player Definition

In [5]:
class HumanPlayer(Player):
    def __init__(self) -> None:
        super().__init__()

    def choose_action(self, game: 'Game') -> tuple[tuple[int, int], Move]:
            available_moves = game.get_possible_moves(game.get_current_player())
            while True:
                row = int(input("Input your action row:"))
                col = int(input("Input your action col:"))
                from_pos = (row, col)
                move = int(input("Input your action move: (1 for top, 2 for bottom, 3 for left, 4 for right):"))
                if move == 1:
                    move = Move.TOP
                elif move == 2:
                    move = Move.BOTTOM
                elif move == 3:
                    move = Move.LEFT
                elif move == 4:
                    move = Move.RIGHT
                else:
                    print("Invalid move, please input again")
                    continue
                
                if (from_pos, move) in available_moves:
                    return from_pos, move
                else:
                    print("Invalid move, please input again")
                    continue

    def give_rew(self, reward):
        pass

    def add_state(self, s):
        pass

## Hyperparameters
---
- ``epochs``: training epochs
- ``alpha``: learning rate
- ``discount_factor``: the discount rate of the Bellman equation;
- ``min_exploration_rate``: the minimum rate for exploration during the training phase
- ``exploration_decay_rate``: the exploration decay rate used during the training
- ``opponent``: the opponent to play against
- ``num_games``: number of games for testing

In [8]:
epochs = 500000
alpha = 0.1
discount_factor = 0.95
min_exploration_rate=0.01
exploration_decay_rate=1e-5
opponent = RandomPlayer()
num_games = 1000

## Let's do some computation

In [ ]:
# create the Q-learning player
q_learning_rl_agent = RLPlayer(
    epochs=epochs,
    alpha=alpha,
    discount_factor=discount_factor,
    min_exploration_rate=min_exploration_rate,
    exploration_decay_rate=exploration_decay_rate,
    opponent=opponent,
)
# train the Q-learning player
q_learning_rl_agent.train()

## Test Reinforcement Learning

In [ ]:
def test(rl_player: 'RLPlayer', random_player, num_games):
    g = Game()
    RLPlayer_wins = 0
    games = 0
    for _ in range(num_games):
        winner = g.play(rl_player, random_player)
        games += 1
        print(games)
        g.reset()
        if winner == 0:
            RLPlayer_wins += 1

    print(f"RLPlayer won {RLPlayer_wins / num_games * 100}%")

test(q_learning_rl_agent, RandomPlayer(), num_games)

### RL Player results

- RL Player 1
    - ``epochs`` = 350000,
    - ``alpha`` = 0.1,
    - ``discount_factor`` = 0.95,
    - ``min_exploration_rate`` = 0.01,
    - ``exploration_decay_rate`` = 1e-5,
     
    **Results**
    - Last 1000 episodes - Mean rewards value: 5.08
    - win rate vs ``RandomPlayer`` in 1000 games - 79%

- RL Player 2
    - ``epochs`` = 500000,
    - ``alpha`` = 0.1,
    - ``discount_factor`` = 0.95,
    - ``min_exploration_rate`` = 0.01,
    - ``exploration_decay_rate`` = 1e-5,
     
    **Results**
    - Last 1000 episodes - Mean rewards value: 4.42
    - win rate vs ``RandomPlayer`` in 1000 games - 82%

- RL Player 3
    - ``epochs`` = 500000,
    - ``alpha`` = 0.1,
    - ``discount_factor`` = 0.95,
    - ``min_exploration_rate`` = 0.01,
    - ``exploration_decay_rate`` = 5e-6,
     
    **Results**
    - Last 1000 episodes - Mean rewards value: 1.72
    - win rate vs ``RandomPlayer`` in 1000 games - 72%
    

## Let's play!

In [ ]:
rl_player = RLPlayer(
    epochs=epochs,
    alpha=alpha,
    discount_factor=discount_factor,
    min_exploration_rate=min_exploration_rate,
    exploration_decay_rate=exploration_decay_rate,
    opponent=opponent,
)
human_player = HumanPlayer()

rl_player.load_policy('RL_player')
g = Game()
winner = g.play(rl_player, human_player, print_flag=True)